# 01 — Experiment Tracking dan Model Registry dengan MLflow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/13-mlops/01_experiment_tracking_mlflow.ipynb)

**MLOps** adalah praktik mengelola seluruh siklus hidup model ML secara terstruktur, seperti DevOps untuk perangkat lunak:

```
data → training → evaluasi → registrasi → deployment → monitoring ─┐
  ▲                                                                 │
  └──────────────── training ulang (data baru / drift) ◀────────────┘
```

Modul-modul sebelumnya mengerjakan setiap kotak secara terpisah. Modul terakhir ini menghubungkannya. Notebook pertama membahas masalah yang pasti kamu alami setelah beberapa minggu bereksperimen:

> *"Model yang akurasinya 0,72 kemarin itu pakai learning rate berapa ya? File mana yang dipakai di produksi: `model_final.pt`, `model_final_v2.pt`, atau `model_final_BENAR.pt`?"*

**Experiment tracking** mencatat setiap percobaan secara otomatis (parameter, metrik, artefak), dan **model registry** mencatat versi model mana yang sedang dipakai. Kita memakai **MLflow** (open source, Apache-2.0, bisa berjalan sepenuhnya lokal).

Isi notebook:
1. Konsep MLflow: experiment, run, parameter, metrik, artefak, tag.
2. Mencatat satu run training (termasuk kurva per epoch).
3. *Hyperparameter sweep* dan memilih model terbaik dengan benar (validasi, bukan test).
4. **Model registry**: mendaftarkan model, alias `champion`, dan memuatnya kembali.

## 0. Instalasi dan Kode

In [ ]:
%pip install -q mlflow torch pandas onnx onnxscript onnxruntime matplotlib

Modul ini memakai kembali model sentimen dari modul 12 ([`12-deployment/app/`](../12-deployment/app/)): `model.py` (tokenizer, model, `Prediktor`) dan `latih.py` (training). Di Colab, file-file itu diunduh otomatis dari GitHub.

In [ ]:
import os
import sys
import urllib.request
from pathlib import Path

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"          # sembunyikan pesan petunjuk MLflow
RAW = "https://raw.githubusercontent.com/riki-profile/ai-engineer-journey/main"

APP12 = Path("../12-deployment/app")                   # jika repo di-clone: pakai file lokal
if not (APP12 / "model.py").exists():                  # di Colab: unduh dari GitHub
    APP12 = Path("app12")
    APP12.mkdir(exist_ok=True)
    for f in ["model.py", "latih.py"]:
        urllib.request.urlretrieve(f"{RAW}/12-deployment/app/{f}", APP12 / f)
if not Path("pyfunc_sentimen.py").exists():
    urllib.request.urlretrieve(f"{RAW}/13-mlops/pyfunc_sentimen.py", "pyfunc_sentimen.py")
sys.path.insert(0, str(APP12.resolve()))

EPOCH = 3 if QUICK_RUN else 30
print(f"Kode modul 12 dari: {APP12.resolve()} | QUICK_RUN: {QUICK_RUN} | epoch: {EPOCH}")

In [ ]:
import mlflow
from mlflow import MlflowClient

# Semua catatan eksperimen disimpan di file SQLite lokal; file artefak di folder mlruns/
mlflow.set_tracking_uri(f"sqlite:///{Path('mlflow.db').resolve()}")
client = MlflowClient()
print("Tracking URI:", mlflow.get_tracking_uri())

## 1. Konsep MLflow Tracking

| Konsep | Arti | Contoh |
|---|---|---|
| **Experiment** | Kelompok percobaan untuk satu masalah | `sentimen-nusax` |
| **Run** | Satu kali training | "dim=64, lr=0.005" |
| **Parameter** | Input run (tidak berubah) | `dim`, `lr`, `epoch`, `seed` |
| **Metrik** | Hasil angka, bisa per langkah | `loss` per epoch, `akurasi_val` |
| **Artefak** | File apa pun | `config.json`, `model.onnx`, grafik |
| **Tag** | Metadata bebas | commit git, nama dataset, catatan |

Tracking bisa disimpan lokal (seperti di sini: file SQLite) atau di **server MLflow** bersama sehingga satu tim bisa melihat semua eksperimen.

## 2. Mencatat Satu Run

Fungsi `latih` dari modul 12 menerima `callback` yang dipanggil setiap epoch. Kita pakai untuk mencatat `loss` dan `akurasi_val` per epoch ke MLflow, sehingga kurva training tersimpan tanpa perlu `print`.

In [ ]:
from latih import latih

mlflow.set_experiment("sentimen-nusax")
FOLDER_RUN = Path("artefak_runs")


def latih_dan_catat(dim=64, lr=5e-3, epoch=EPOCH, seed=0, nama=None, bersarang=False):
    """Latih satu model dan catat semuanya ke MLflow. Mengembalikan run_id."""
    with mlflow.start_run(run_name=nama or f"dim{dim}-lr{lr}", nested=bersarang) as run:
        mlflow.log_params({"dim": dim, "lr": lr, "epoch": epoch, "seed": seed, "bahasa": "indonesian"})
        mlflow.set_tags({"model": "embedding+MLP", "data": "NusaX-Senti indonesian"})
        folder = FOLDER_RUN / run.info.run_id

        def catat(ep, loss, akurasi_val):
            mlflow.log_metrics({"loss": loss, "akurasi_val_epoch": akurasi_val}, step=ep)

        config = latih(folder, epoch=epoch, dim=dim, lr=lr, seed=seed, callback=catat)
        mlflow.log_metrics({"akurasi_val": config["akurasi_val"], "akurasi_test": config["akurasi_test"],
                            "n_vocab": config["n_vocab"]})
        mlflow.log_artifacts(str(folder), artifact_path="artefak")     # config, vocab, model.onnx, model.pt
        return run.info.run_id


run_id = latih_dan_catat(nama="percobaan-pertama")
run = client.get_run(run_id)
print("Parameter:", run.data.params)
print("Metrik   :", {k: round(v, 4) for k, v in run.data.metrics.items()})
print("Artefak  :", [a.path for a in client.list_artifacts(run_id, "artefak")])

In [ ]:
import matplotlib.pyplot as plt

riwayat = {m: client.get_metric_history(run_id, m) for m in ["loss", "akurasi_val_epoch"]}
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for a, (nama, titik) in zip(ax, riwayat.items()):
    a.plot([t.step for t in titik], [t.value for t in titik], "o-")
    a.set(title=nama, xlabel="epoch")
plt.tight_layout()
plt.show()

## 3. Hyperparameter Sweep

Kita coba beberapa kombinasi `dim` (ukuran embedding) dan `lr` (learning rate). Semua run dikelompokkan di bawah satu **run induk** (`nested=True`) agar rapi.

**Aturan penting memilih model:** pilih berdasarkan **akurasi validasi**, bukan akurasi test. Jika kita memilih berdasarkan test, test set ikut "dipakai untuk training" (lewat pilihan kita), dan angka test tidak lagi jujur untuk memperkirakan performa di data baru. Test hanya dilihat sekali, untuk model yang sudah terpilih.

In [ ]:
import itertools

import pandas as pd

GRID = {"dim": [32, 64], "lr": [5e-3]} if QUICK_RUN else {"dim": [32, 64, 128], "lr": [1e-3, 5e-3, 1e-2]}

with mlflow.start_run(run_name="sweep-dim-lr") as induk:
    for dim, lr in itertools.product(GRID["dim"], GRID["lr"]):
        latih_dan_catat(dim=dim, lr=lr, bersarang=True)
        print(f"selesai: dim={dim}, lr={lr}")

hasil = mlflow.search_runs(experiment_names=["sentimen-nusax"],
                           filter_string=f"tags.mlflow.parentRunId = '{induk.info.run_id}'",
                           order_by=["metrics.akurasi_val DESC"])
tabel = hasil[["run_id", "params.dim", "params.lr", "metrics.akurasi_val", "metrics.akurasi_test"]]
tabel

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for _, baris in hasil.iterrows():
    titik = client.get_metric_history(baris["run_id"], "akurasi_val_epoch")
    ax.plot([t.step for t in titik], [t.value for t in titik], label=f"dim={baris['params.dim']}, lr={baris['params.lr']}")
ax.set(title="Akurasi validasi per epoch", xlabel="epoch", ylabel="akurasi")
ax.legend(fontsize=8)
plt.show()

terbaik = hasil.iloc[0]
print(f"Terpilih (akurasi validasi tertinggi): dim={terbaik['params.dim']}, lr={terbaik['params.lr']} | "
      f"val {terbaik['metrics.akurasi_val']:.3f} | test {terbaik['metrics.akurasi_test']:.3f}")

Validasi NusaX hanya 100 kalimat, jadi selisih satu-dua kalimat (1–2 poin persen) antar-konfigurasi bisa jadi hanya kebetulan. Notebook 02 mengukur seberapa besar variasi akibat *seed* acak, supaya kita tidak "memenangkan" konfigurasi karena keberuntungan.

## 4. Model Registry

Experiment tracking menjawab "apa yang sudah dicoba". **Model registry** menjawab "model mana yang resmi dipakai":
- Setiap model terdaftar punya **versi** (1, 2, 3, ...), masing-masing terhubung ke run yang menghasilkannya (lineage).
- **Alias** seperti `champion` (dipakai produksi) atau `challenger` (kandidat) menunjuk ke satu versi. Aplikasi memuat `models:/sentimen-id@champion`; mengganti model cukup dengan memindahkan alias, tanpa mengubah kode aplikasi.

Model dicatat sebagai **pyfunc** MLflow: antarmuka standar `predict(DataFrame)` yang bisa dimuat oleh alat MLflow mana pun (termasuk `mlflow models serve` untuk langsung menyajikan REST API). Pembungkusnya ada di [`pyfunc_sentimen.py`](pyfunc_sentimen.py):

In [ ]:
print(Path("pyfunc_sentimen.py").read_text())

In [ ]:
NAMA_MODEL = "sentimen-id"
folder_terbaik = FOLDER_RUN / terbaik["run_id"]

with mlflow.start_run(run_id=terbaik["run_id"]):          # tambahkan model ke run terbaik
    info = mlflow.pyfunc.log_model(
        name="model",
        python_model="pyfunc_sentimen.py",                 # models from code
        artifacts={"artefak": str(folder_terbaik)},
        code_paths=[str(APP12 / "model.py")],
        input_example=pd.DataFrame({"teks": ["Barangnya bagus sekali"]}),
        pip_requirements=["onnxruntime", "numpy", "pandas"],
        registered_model_name=NAMA_MODEL,
    )

versi = client.search_model_versions(f"name='{NAMA_MODEL}'")
versi_baru = max(versi, key=lambda v: int(v.version))
client.set_registered_model_alias(NAMA_MODEL, "champion", versi_baru.version)
client.update_model_version(NAMA_MODEL, versi_baru.version,
                            description=f"Terbaik dari sweep; akurasi val {terbaik['metrics.akurasi_val']:.3f}")
print(f"Model '{NAMA_MODEL}' versi {versi_baru.version} → alias 'champion' (run {versi_baru.run_id[:8]})")

Aplikasi (misalnya API modul 12) cukup memuat model lewat alias. Siapa pun yang memuatnya mendapat model yang sama, lengkap dengan kode preprocessing dan dependensinya.

In [ ]:
champion = mlflow.pyfunc.load_model(f"models:/{NAMA_MODEL}@champion")
champion.predict(pd.DataFrame({"teks": ["Pengirimannya cepat, mantap!", "Pelayanannya buruk sekali, saya kecewa.",
                                        "Paket sampai hari Senin."]}))

**Melihat semuanya di UI.** Di komputer lokal, jalankan dari folder ini:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

lalu buka `http://127.0.0.1:5000`: tabel semua run, grafik perbandingan metrik, artefak, dan halaman model registry. Di Colab, UI ini tidak bisa dibuka langsung; gunakan `mlflow.search_runs` seperti di atas, atau server MLflow bersama.

**Alternatif:** *Weights & Biases* (layanan cloud, UI kolaboratif yang sangat baik, gratis untuk penggunaan pribadi, butuh API key yang disimpan di `.env`/Secrets), *Comet*, *Neptune*, atau MLflow yang di-hosting di Databricks/cloud. Konsepnya sama: run, parameter, metrik, artefak, registry.

## Ringkasan

- **Experiment tracking** mencatat setiap run: parameter, metrik (termasuk per epoch), artefak, dan tag, sehingga eksperimen bisa dibandingkan dan diulang.
- **Sweep** hyperparameter: kelompokkan dengan run induk; pilih berdasarkan **validasi**, laporkan **test** hanya untuk model terpilih.
- **Model registry**: versi + alias (`champion`, `challenger`). Aplikasi memuat model lewat alias, sehingga promosi/rollback cukup dengan memindahkan alias.
- **pyfunc** membungkus model beserta preprocessing-nya dalam antarmuka standar.

## Latihan

1. **Sweep lebih luas.** Tambahkan hyperparameter `seed` (0, 1, 2) ke sweep untuk konfigurasi terbaik dan dua konfigurasi di bawahnya. Apakah urutan "terbaik" tetap sama untuk semua seed? Apa artinya bagi keputusan memilih model?
2. **Artefak grafik.** Di `latih_dan_catat`, buat confusion matrix pada data validasi, simpan sebagai PNG, dan catat dengan `mlflow.log_figure`. Bandingkan confusion matrix dua run di UI MLflow.
3. **Rollback.** Daftarkan model dari run terburuk sweep sebagai versi baru, pindahkan alias `champion` ke versi itu, lalu kembalikan (rollback) ke versi sebelumnya. Tulis fungsi `promosikan(nama_model, versi)` yang mencatat siapa dan kapan alias dipindah sebagai tag di versi model.